## Alterada a referência de eNB de MeContext para ManagedElement

# Processando os dados do MO TDD 

In [ ]:
# Função para carregar os arquivos
base_path = r"C:\Dumps\todos_enms"

def load_parquet(pattern, columns=None):
    import glob, pandas as pd
    files = glob.glob(pattern)
    if not files:
        return pd.DataFrame()
    df = pd.concat((pd.read_parquet(f, columns=columns) for f in files),
                   ignore_index=True)
    try:
        df = df.convert_dtypes(dtype_backend="pyarrow")
    except Exception:
        pass
    return df


In [ ]:
%%time
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\EUtranCellTDD_*_part*.parquet")
df1['eNB'] = df1['ManagedElement']
# Procura coluna CELL
cell_cols = [col for col in df1.columns if 'UtranCellTDDId' in col]
df1['CELL'] = df1[cell_cols].bfill(axis=1).iloc[:, 0] if len(cell_cols) > 1 else (df1[cell_cols[0]] if cell_cols else None)

In [ ]:
# Extraindo os últimos 2 caracteres do nome do eNB para determinar o UF
df1['UF'] = df1['eNB'].str[-2:]
# Calculando o PCI com base em 'physicalLayerCellIdGroup' e 'physicalLayerSubCellId'
df1['PCI'] = df1['physicalLayerCellIdGroup'].astype(int)*3 + df1['physicalLayerSubCellId'].astype(int)
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth', None)
TDD= df1.copy()
TDD.head(2)

# Pegando o eNBId e o PLMN do MO ENodeBFunction

In [ ]:
ENodeBFunction= pd.read_csv('C:/Dumps/LTE_total/ENodeBFunction.txt', delimiter='\t', engine='python', usecols=['UF', 'eNB','eNBId', 'eNodeBPlmnId_mcc','eNodeBPlmnId_mnc'])
ENodeBFunction.head()

# fazendo o merge com o TDD para incluir as informações do ENodeBFunction no TDD
df1= pd.merge(df1, ENodeBFunction, on=['UF', 'eNB'],how='inner')
df1.head()

In [ ]:
# Preparando os dados para criar a coluna CGI
df1['eNodeBPlmnId_mcc'] = df1['eNodeBPlmnId_mcc'].fillna(0).astype(int).astype(str)
df1['eNodeBPlmnId_mnc'] = df1['eNodeBPlmnId_mnc'].fillna(0).astype(int).astype(str)
df1['eNBId'] = df1['eNBId'].fillna(0).astype(int).astype(str)
df1['cellId'] = df1['cellId'].fillna(0).astype(int).astype(str)

# Criando a coluna CGI
df1['CGI'] = df1['eNodeBPlmnId_mcc'] + df1['eNodeBPlmnId_mnc'] + '-' + df1['eNBId'] + '-' + df1['cellId']
df1.head()

In [ ]:
# Selecionando colunas específicas para formar um novo DataFrame df2
df2 = df1.loc[:, ['UF', 'eNB', 'eNBId', 'CELL', 'cellId','earfcn', 'channelBandwidth', 'tac', 'PCI', 'physicalLayerCellIdGroup', 'physicalLayerSubCellId',
                  'CGI','crsGain', 'cellRange', 'primaryPlmnReserved', 'primaryUpperLayerInd','rachRootSequence','administrativeState', 'operationalState']]

# Criando uma coluna 'Site' baseada na concatenação do prefixo do eNB com o UF
# v2: o site fisico vem do nome da CELULA, nao do no. Preventivo: hoje as 1.370
# celulas TDD tem CELL[1:4] == eNB[1:4], mas se um Hotel de BTS passar a servir
# TDD a chave pelo no faria o hospede herdar a coordenada do hospedeiro.
df2['Site'] = df2['CELL'].str[1:4] + '_' + df2['UF']

# Removendo linhas onde 'cellId' é NaN
df2.dropna(subset=['cellId'], inplace=True)

# Convertendo colunas do tipo float para int quando todos os valores são inteiros
df2 = df2.apply(lambda col: pd.to_numeric(col, errors='coerce').astype('Int64') if col.dtype == 'float64' else col)

# Exibindo as primeiras linhas do DataFrame final para verificação

df2.head()

# Usando o eutrancellfdd para obter os dados fisicos do TDD

In [ ]:
# Abrindo o arquivo eutrancellfdd selecionando apenas as colunas necessárias
colunas = [
    "UF", "CN", "MUNICIPIO", "BAIRRO", "ENDERECO", 'Site_Name', "eNB", "eNBId", "CELL",
    "cellId", "earfcndl", "dlChannelBandwidth", "tac", "PCI",
    "physicalLayerCellIdGroup", "physicalLayerSubCellId", 'rachRootSequence','catm1SupportEnabled',"CGI", "crsGain",
    "cellRange", "primaryPlmnReserved", 'primaryUpperLayerInd',"Latitude", "Longitude", "Azimute",
    "Altura", "administrativeState", "operationalState", "SiteType", 'Site_Name'
]

eutrancellfdd= pd.read_csv('C:/Camilo/EUtranCellFDD.csv', usecols=colunas,engine='python', encoding="ISO-8859-1")

# Fazendo uma copia do fdd para usar mais adiante no join com o TDD 
fdd= eutrancellfdd.copy()
fdd['Tipo']= 'FDD'

eutrancellfdd['Setor']= eutrancellfdd['CELL'].str[-1].replace(['4','5','6', '7', '8', '9'], ['1','2','3','1','2','3'])

# Selecionando apenas as colunas necessarias para o join com o TDD
eutrancellfdd= eutrancellfdd.loc[:,['UF', 'CN','eNB','CELL','MUNICIPIO', 'BAIRRO', 'ENDERECO', 'Site_Name', 'Setor','Latitude', 'Longitude', 'Azimute','Altura','SiteType']]
# v2: site fisico pelo nome da CELULA (mesma chave do lado TDD, celula 8).
eutrancellfdd['Site']= eutrancellfdd['CELL'].str[1:4] + '_' + eutrancellfdd['UF']
eutrancellfdd.drop(columns=['eNB','CELL'],inplace=True)
# v2: o dedup TEM de ser na chave do merge (['UF','Site','Setor']). Estava em
# ['Site_Name','Setor'], que NAO e a chave: duas linhas com mesmo Site/Setor e
# Site_Name diferente sobreviviam as duas e o join virava muitos-para-muitos,
# duplicando celula TDD. Nao aparecia porque o EUtranCellFDD vinha com o
# Site_Name homogeneizado pelo bug da chave do no; com o 4_MOs_Camilo corrigido,
# 619 baldes passariam a casar mais de uma linha. Medido em 2026-09-02.
eutrancellfdd.drop_duplicates(subset=['UF','Site','Setor'],inplace=True)
eutrancellfdd.head(2)

In [ ]:
df2['Setor']= df2['CELL'].str[-1].replace(['4','5','6', '7', '8', '9'], ['1','2','3','1','2','3'])
df2.head()

# Adicionando as informações fisicas ao TDD

In [ ]:
# Fazendo o merge do TDD com as informações fisicas obtidas do eutrancellfdd

tdd= pd.merge(df2, eutrancellfdd, on=['UF','Site', 'Setor'], how='left')
# pd.set_option('display.max_columns',None)
tdd['Tipo']= 'TDD'
tdd.head()

In [ ]:
# Organizando as colunas padronizando TDD com o FDD
colunas = [
    "UF", "CN", "MUNICIPIO", "BAIRRO", "ENDERECO", 'Site_Name', "eNB", "eNBId", "CELL",
    "cellId", "earfcn", "channelBandwidth", "tac", "PCI",
    "physicalLayerCellIdGroup", "physicalLayerSubCellId", "CGI", "crsGain",
    "cellRange", "primaryPlmnReserved",'primaryUpperLayerInd', 'rachRootSequence',"Latitude", "Longitude", "Azimute",
    "Altura", "administrativeState", "operationalState", "SiteType", "Tipo"
]

tdd= tdd.loc[:, colunas]
tdd.head(2)

# Criando o arquivo TDD_FDD com a concatenação do tdd e fdd

In [ ]:
# primeiro é necessario renomear as colunas do TDD para ficarem aderentes ao FDD
tdd.rename(columns={'earfcn': 'earfcndl', 'channelBandwidth':'dlChannelBandwidth'},inplace=True)
tdd_fdd= pd.concat([fdd, tdd])
tdd_fdd['Regional']= tdd_fdd['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

# Usando o método pop para remover a coluna 'Regional' e armazená-la em uma variável
regional_col = tdd_fdd.pop('Regional')

# Inserindo essa coluna de volta no DataFrame na primeira posição (índice 0)
tdd_fdd.insert(0, 'Regional', regional_col)

tdd_fdd['Site']= tdd_fdd['eNB'].str[1:4]

tdd_fdd.sort_values(by=['Regional', 'UF', 'Site', 'eNB','CELL'])
tdd_fdd.drop(columns='Site', inplace=True)
tdd_fdd.dropna(subset='administrativeState',inplace=True)
tdd_fdd.head(2)

In [ ]:
from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

arquivo_MoB_IMP= 'C:/Vivo Reports/MoB_IMP/' + 'EUtranCell_TDD_FDD'+'.txt'
arquivo_txt= 'C:/Camilo/' + 'EUtranCell_TDD_FDD'+'.txt'
arquivo_csv= 'C:/Camilo/' + 'EUtranCell_TDD_FDD'+ '_'+str(semana_atual)+'.csv'
arquivo= 'C:/Camilo/' + 'EUtranCell_TDD_FDD'+'.csv'
arquivo_semana= 'C:/Camilo/' + 'EUtranCell_TDD_FDD'+ '_'+str(semana_atual)+'.txt'

tdd_fdd.to_csv(arquivo_txt,sep='\t', index= None, errors='ignore')
tdd_fdd.to_csv(arquivo_csv, index=None, encoding='ISO-8859-1', errors='ignore')
tdd_fdd.to_csv(arquivo, index=None, encoding='ISO-8859-1', errors='ignore')
tdd_fdd.to_csv(arquivo_semana,sep='\t', index=None, errors='ignore')
tdd_fdd.to_csv(arquivo_MoB_IMP,sep='\t', index= None, errors='ignore')

print('Ok')

# Importando a biblioteca pandas
import pandas as pd

# Definindo o caminho do arquivo de entrada (nota: certifique-se de que a variável semana_atual esteja definida)
#arquivo_txt = 'C:/Camilo/' + 'EUtranCell_TDD_FDD' + '_' + str(semana_atual) + '.txt'

# Lendo o arquivo de texto no DataFrame df
df = pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.txt', delimiter='\t', engine='python', encoding='ISO-8859-1')

# Iterando através de cada linha no DataFrame
for i, row in df.iterrows():
    # Verificando se a coluna MUNICIPIO está em branco
    if pd.isna(row['MUNICIPIO']):
        # Selecionando as linhas que têm a mesma eNB e MUNICIPIO não vazio
        municipio_info = df[(df['eNB'] == row['eNB']) & df['MUNICIPIO'].notna()]
        # Se encontrou linhas correspondentes, preencha os valores em branco
        if not municipio_info.empty:
            df.at[i, 'MUNICIPIO'] = municipio_info['MUNICIPIO'].values[0]
            df.at[i, 'BAIRRO'] = municipio_info['BAIRRO'].values[0]
            df.at[i, 'ENDERECO'] = municipio_info['ENDERECO'].values[0]
            df.at[i, 'Latitude'] = municipio_info['Latitude'].values[0]
            df.at[i, 'Longitude'] = municipio_info['Longitude'].values[0]
            df.at[i, 'Altura'] = municipio_info['Altura'].values[0]

    # Verificando se a coluna Azimute está em branco
    if pd.isna(row['Azimute']):
        # Obtendo o último dígito da coluna CELL
        cell_last_digit = str(row['CELL'])[-1]
        # Selecionando as linhas que têm a mesma eNB, mesmo último dígito da CELL e Azimute não vazio
        azimute_info = df[(df['eNB'] == row['eNB']) & (df['CELL'].astype(str).str[-1] == cell_last_digit) & df['Azimute'].notna()]
        # Se encontrou linhas correspondentes, preencha o valor Azimute em branco
        if not azimute_info.empty:
            df.at[i, 'Azimute'] = azimute_info['Azimute'].values[0]

# Removendo linhas duplicadas
df = df.drop_duplicates()

# Definindo os caminhos dos arquivos de saída (em formatos TXT e CSV)
arquivo_txt = 'C:/Camilo/' + 'EUtranCell_TDD_FDD' + '_' + str(semana_atual) + '.txt'
arquivo_txt2= 'C:/Camilo/' + 'EUtranCell_TDD_FDD'+'.csv'
arquivo_csv = 'C:/Camilo/' + 'EUtranCell_TDD_FDD' + '_' + str(semana_atual) + '.csv'
arquivo_compartilhado= 'C:/Vivo Reports/LTE/LTE MOs/' + 'EUtranCell_TDD_FDD'+ '_'+str(semana_atual)+'.csv'
#df.drop(['Unnamed: 0'], axis='columns', inplace=True)

# Salvando o DataFrame nos arquivos 
df.to_csv(arquivo_txt, sep='\t', encoding='ISO-8859-1', index=None)
df.to_csv(arquivo_txt2, encoding='ISO-8859-1', index=None)
df.to_csv(arquivo_csv, encoding='ISO-8859-1', index=None)
df.to_csv(arquivo_compartilhado, encoding='ISO-8859-1', index=None)
# Imprimindo "Finished" no console para indicar que o script foi concluído com sucesso
print('Finished')

In [ ]:
import pandas as pd

arquivo_csv= 'C:/Camilo/' + 'EUtranCell_TDD_FDD'+ '_'+str(semana_atual)+'.csv'


df= pd.read_csv(arquivo_csv,engine='python', encoding='ISO-8859-1')

df['Banda']= df['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'O', 'L'],
                                         ['2600','2600', '1800', '700', '2100', '2600', '850','1800', '2300', '2300'])

df.rename(columns={'Latitude': 'LATITUDE', 'Longitude': 'LONGITUDE'},inplace= True)

df.drop(['physicalLayerCellIdGroup', 'physicalLayerSubCellId', 'primaryPlmnReserved'], axis='columns', inplace=True)

cellref= pd.read_excel('C:/Camilo/cellref.xlsx')

cellref1= cellref.loc[:, ['SiteID','Sector', 'Site_Name','Azimuth', 'Height']]

cellref1.rename(columns={'Azimuth': 'AZIMUTE', 'Height': 'ALTURA', 'Site_Name': 'NOME_SITE'},inplace= True)


df2= pd.merge(df,cellref1, left_on=['eNB', 'CELL'], right_on= ['SiteID', 'Sector'], how= 'left')

df3= df2.loc[:, ['UF', 'CN','MUNICIPIO', 'BAIRRO','ENDERECO','NOME_SITE','eNB', 'eNBId', 'CELL','cellId','Banda', 'earfcndl', 
                 'dlChannelBandwidth', 'tac', 'PCI','crsGain','administrativeState', 'operationalState','cellRange', 'AZIMUTE','LATITUDE',
                 'LONGITUDE', 'ALTURA', 'SiteType']]

df4= df3.set_index('UF')

df4.drop_duplicates(subset=['eNB', 'CELL'], inplace= True)

# errors= 'ignore' utilizado para salvar o encoding encoding='ISO-8859-1' para csv sem erros

arquivo_txt= 'C:/Camilo/' + 'DT_Info_LTE'+ '_'+str(semana_atual)+'.txt'
arquivo_csv= 'C:/Camilo/' + 'DT_Info_LTE'+ '_'+str(semana_atual)+'.csv'
arquivo_compartilhado= 'C:/Vivo Reports/LTE/LTE MOs/' + 'DT_Info_LTE'+ '_'+str(semana_atual)+'.csv'

df4.to_csv(arquivo_txt, sep='\t')
df4.to_csv(arquivo_csv, encoding='ISO-8859-1', errors='ignore')
df.to_csv(arquivo_compartilhado, encoding='ISO-8859-1', index=None)

print('Finished')

# Arquivo Tilt por setor

In [ ]:
import pandas as pd
import numpy as np
import re

# Abrindo o arquivo RetSubunit
df= pd.read_csv('C:/Dumps/LTE_total/RetSubunit.txt', sep='\t', low_memory=False)

# Criando a coluna 'Regional'
df['Regional'] = df['UF'].replace(['AM', 'AP','MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

# Criando as colunas 'Tilt_700', 'Tilt_1800', 'Tilt_2600', 'Tilt_2100', 'Tilt_2300'
df['Tilt_700'] = np.where(df['userLabel'].str.contains('7'), df['electricalAntennaTilt'], -1000)
df['Tilt_1800'] = np.where(df['userLabel'].str.contains('L18'), df['electricalAntennaTilt'], -1000)
df['Tilt_2600'] = np.where(df['userLabel'].str.contains('L26'), df['electricalAntennaTilt'], -1000)
df['Tilt_2100'] = np.where(df['userLabel'].str.contains('L21'), df['electricalAntennaTilt'], -1000)
df['Tilt_2300'] = np.where(df['userLabel'].str.contains('23'), df['electricalAntennaTilt'], -1000)

# Criando a coluna 'Sector' com base nos valores da coluna 'userLabel' e verificando os setores 4/5/6.
def get_sector(userLabel):
    # Verifica setores definidos por tags
    match = re.search(r'<D0(\d)>', userLabel)
    if match:
        return match.group(1)
    # Caso contrário, assume que o setor é o número após o terceiro "_"
    parts = userLabel.split('_')
    if len(parts) > 2 and parts[2].isdigit():
        return parts[2]
    return ''

# Convertendo 'userLabel' to string
df['userLabel'] = df['userLabel'].astype(str)
df['Sector'] = df['userLabel'].apply(get_sector)

# Agrupando por 'UF', 'eNB', 'Sector' e obtenha o valor máximo para as colunas 'Tilt...'
grouped = df.groupby(['Regional','UF', 'eNB', 'Sector']).agg(
    Tilt_700=('Tilt_700', 'max'),
    Tilt_1800=('Tilt_1800', 'max'),
    Tilt_2600=('Tilt_2600', 'max'),
    Tilt_2100=('Tilt_2100', 'max'),
    Tilt_2300=('Tilt_2300', 'max')).reset_index()
grouped.replace(-1000, '', inplace=True)

# Abrindo o arquivo EutranCell para pegar informação de MUNICIPIO
EutranCell = pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.csv', encoding='ISO-8859-1', usecols=['MUNICIPIO', 'eNB', 'CELL'])
# Contar o número de células por 'MUNICIPIO' e 'eNB' e adicionar como uma nova coluna 'Cell_Count'
EutranCell = EutranCell.groupby(['MUNICIPIO', 'eNB'])['CELL'].count().reset_index().rename(columns={'CELL': 'Cell_Count'})
EutranCell.drop(columns=['Cell_Count'], inplace= True)
junta= pd.merge(grouped, EutranCell, left_on= 'eNB', right_on= 'eNB', how= 'left')
# Organizando o dataframe para MUNICIPIO ficar como a segunda coluna
cols = junta.columns.tolist()
cols.remove('MUNICIPIO')
cols = ['Regional', 'MUNICIPIO'] + [col for col in cols if col != 'Regional']
junta = junta.reindex(columns=cols)
junta['Sector'] = junta['Sector'].astype(str).str.strip()
junta['Sector'] = junta['Sector'].replace('', np.nan)
junta.dropna(subset=['Sector'], inplace=True)

junta.to_csv('C:/Vivo Reports/LTE/LTE MOs/tilt_setor_LTE.csv', index=None)
print('OK')

### TermPointToENB 4G--4G - Importante para o CA entre nodes do mesmo site

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\TermPointToENB_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['TermPointToENBId'] = df1['termPointToENBId'].combine_first(df1['TermPointToENBId'])

df1.drop('termPointToENBId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

# Pegando os dados de EUtranCell para identificarmos a ancora
EutranCell= pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.txt', sep='\t', encoding='ISO-8859-1', usecols=['MUNICIPIO','eNB', 'CGI'])

EutranCell['CGI'] = EutranCell['CGI'].str.extract(r'([^\-]*\-[^\-]*)')

EutranCell.drop_duplicates(inplace=True)
EutranCell.rename(columns={'eNB': 'Ancora'},inplace=True)
EutranCell.head()# Pegando os dados de EUtranCell para identificarmos a ancora
EutranCell= pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.txt', sep='\t', encoding='ISO-8859-1', usecols=['MUNICIPIO','eNB', 'CGI'])

EutranCell['CGI'] = EutranCell['CGI'].str.extract(r'([^\-]*\-[^\-]*)')

EutranCell.drop_duplicates(inplace=True)
EutranCell.rename(columns={'eNB': 'Ancora'},inplace=True)

# Fazendo o join do TermpointToENB com o EUtranCell para obtermos o eNB_Name da Ancora
df2= pd.merge(df1, EutranCell, left_on=['TermPointToENBId'], right_on=['CGI'], how='left')
df2.drop_duplicates(subset=['UF', 'eNB', 'TermPointToENBId'], inplace=True)

df2['Regional']= df2['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N','N','N','N','N', 'BASE', 'BASE'])

# Primeiro, obtemos todas as colunas do DataFrame
colunas = df2.columns.tolist()

# Extraímos as duas primeiras colunas
primeiras_duas = colunas[:2]

# Criamos a nova ordem de colunas, começando com as duas primeiras e depois as colunas especificadas
# Asseguramos que as primeiras duas colunas não sejam repetidas no final
colunas_novas = primeiras_duas + ["Regional", "UF", "MUNICIPIO", "eNB", "Ancora", "CGI"] + [col for col in colunas[2:] if col not in ["Regional", "UF", "MUNICIPIO", "eNB", "Ancora", "CGI"]]

# Aplicamos a nova ordem ao DataFrame
df2 = df2[colunas_novas]

arquivo_txt= 'C:/Camilo/' + 'TermPointToENB'+ '_'+str(semana_atual)+'.txt'
arquivo_csv= 'C:/Camilo/' + 'TermPointToENB'+ '_'+str(semana_atual)+'.csv'

df2.to_csv(arquivo_txt, sep='\t', index=None)
df2.to_csv(arquivo_csv, encoding='ISO-8859-1', errors='ignore',index=None)

print('Done')

# Conta Setores por Carrier/eNB

import pandas as pd

EUtranCell_TDD_FDD= pd.read_csv('C:/Vivo Reports/LTE/LTE MOs/EUtranCell_TDD_FDD_47.csv', encoding= 'ISO-8859-1')
EUtranCell_TDD_FDD['Site']= EUtranCell_TDD_FDD['eNB'].str[1:4] + EUtranCell_TDD_FDD['eNB'].str[-3:] 
EUtranCell_TDD_FDD.head()


# Passo 1: Agrupar por 'eNB' e contar os valores distintos de 'earfcndl' para cada grupo
num_carrier_por_enb = EUtranCell_TDD_FDD.groupby('Site')['earfcndl'].nunique().reset_index(name='NUM_CARRIER')
num_cells_por_freq = EUtranCell_TDD_FDD.groupby(['Site','earfcndl'])['CELL'].nunique().reset_index(name='NUM_CELL_FREQ')

# Passo 2: Mapear a contagem de volta ao DataFrame original
# Isso cria um mapeamento de 'eNB' para 'NUM_CARRIER'
mapeamento_num_carrier = dict(zip(num_carrier_por_enb['Site'], num_carrier_por_enb['NUM_CARRIER']))
mapeamento_num_cell = dict(zip(zip(num_cells_por_freq['Site'], num_cells_por_freq['earfcndl']), num_cells_por_freq['NUM_CELL_FREQ']))

# Passo 3: Aplicar o mapeamento para criar a nova coluna 'NUM_CARRIER'
EUtranCell_TDD_FDD['NUM_CARRIER'] = EUtranCell_TDD_FDD['Site'].map(mapeamento_num_carrier)
EUtranCell_TDD_FDD['NUM_CELL_FREQ'] = EUtranCell_TDD_FDD.apply(lambda row: mapeamento_num_cell[(row['Site'], row['earfcndl'])], axis=1)

EUtranCell_TDD_FDD1= EUtranCell_TDD_FDD[['UF', 'CN', 'MUNICIPIO', 'Site','eNB', 'eNBId', 'earfcndl', 'NUM_CARRIER', 'NUM_CELL_FREQ']]
EUtranCell_TDD_FDD1= EUtranCell_TDD_FDD1.copy()
EUtranCell_TDD_FDD1.drop_duplicates(subset=['Site', 'eNB', 'earfcndl'],inplace=True)
EUtranCell_TDD_FDD1['Regional']= EUtranCell_TDD_FDD1['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE']) 

EUtranCell_TDD_FDD1.sort_values(by=['Regional','UF', 'Site', 'eNB', 'eNBId', 'earfcndl'], ascending=[True,True, True, True, True, True])

# Passo 1: Obter a lista de colunas, excluindo 'Regional' se já estiver na lista
colunas = [col for col in EUtranCell_TDD_FDD1.columns if col != 'Regional']

# Passo 2: Inserir 'Regional' no início da lista
colunas.insert(0, 'Regional')

# Passo 3: Reordenar as colunas no DataFrame usando a nova lista de colunas
EUtranCell_TDD_FDD1 = EUtranCell_TDD_FDD1[colunas]

# O DataFrame EUtranCell_TDD_FDD1 agora tem 'Regional' como a primeira coluna


# Seu DataFrame EUtranCell_TDD_FDD agora tem uma nova coluna 'NUM_CARRIER' com a contagem desejada
EUtranCell_TDD_FDD1.to_excel('C:/Vivo Reports/LTE/LTE MOs/Conta_Carrier_e_Cells.xlsx', index=None)

# Script que preenche os campos

# Importando a biblioteca pandas
"""
import pandas as pd

# Definindo o caminho do arquivo de entrada (nota: certifique-se de que a variável semana_atual esteja definida)
arquivo_txt = 'C:/Camilo/' + 'EUtranCellFDD' + '_' + str(semana_atual) + '.txt'

# Lendo o arquivo de texto no DataFrame df
df = pd.read_csv(arquivo_txt, delimiter='\t', engine='python', encoding='ISO-8859-1')

# Iterando através de cada linha no DataFrame
for i, row in df.iterrows():
    # Verificando se a coluna MUNICIPIO está em branco
    if pd.isna(row['MUNICIPIO']):
        # Selecionando as linhas que têm a mesma eNB e MUNICIPIO não vazio
        municipio_info = df[(df['eNB'] == row['eNB']) & df['MUNICIPIO'].notna()]
        # Se encontrou linhas correspondentes, preencha os valores em branco
        if not municipio_info.empty:
            df.at[i, 'MUNICIPIO'] = municipio_info['MUNICIPIO'].values[0]
            df.at[i, 'BAIRRO'] = municipio_info['BAIRRO'].values[0]
            df.at[i, 'ENDERECO'] = municipio_info['ENDERECO'].values[0]
            df.at[i, 'Latitude'] = municipio_info['Latitude'].values[0]
            df.at[i, 'Longitude'] = municipio_info['Longitude'].values[0]
            df.at[i, 'Altura'] = municipio_info['Altura'].values[0]

    # Verificando se a coluna Azimute está em branco
    if pd.isna(row['Azimute']):
        # Obtendo o último dígito da coluna CELL
        cell_last_digit = str(row['CELL'])[-1]
        # Selecionando as linhas que têm a mesma eNB, mesmo último dígito da CELL e Azimute não vazio
        azimute_info = df[(df['eNB'] == row['eNB']) & (df['CELL'].astype(str).str[-1] == cell_last_digit) & df['Azimute'].notna()]
        # Se encontrou linhas correspondentes, preencha o valor Azimute em branco
        if not azimute_info.empty:
            df.at[i, 'Azimute'] = azimute_info['Azimute'].values[0]

# Removendo linhas duplicadas
df = df.drop_duplicates()

# Definindo os caminhos dos arquivos de saída (em formatos TXT e CSV)
arquivo_txt = 'C:/Camilo/' + 'EUtranCellFDD' + '_' + str(semana_atual) + '.txt'
arquivo_csv = 'C:/Camilo/' + 'EUtranCellFDD' + '_' + str(semana_atual) + '.csv'

# Salvando o DataFrame nos arquivos 
df.to_csv(arquivo_txt, sep='\t')
df.to_csv(arquivo_csv, encoding='ISO-8859-1')

# Imprimindo "Finished" no console para indicar que o script foi concluído com sucesso
print('Finished')
"""